In [7]:
def tot(data):
    # 범주형 변수 -> 최빈값으로 결측치를 보간하는 함수
    def fillna_col(columns, data):
        tmp = data[columns].value_counts(dropna=True)
        max_v = max(data[columns].value_counts(dropna=True))
        max_c = tmp.loc[tmp==max_v].index[0]
        data[columns] = data[columns].fillna(max_c)
        return data
    # 숫자형 변수 -> 선형보간하는 함수
    def fillna_val(columns, data):
        data[columns] = data[columns].interpolate(method='linear')
        return data
    
    def fillna_total(data):
        tmp_n = data.isna().sum()
        nan_list = list(tmp_n[tmp_n>0].index)
        for columns in nan_list:
            if data[columns].dtype == 'O':
                fillna_col(columns, data)
            else:
                fillna_val(columns, data)
        return data
    data = fillna_total(data)
    def add_var(data):
        # 준공연도
        data['준공일자'] = data['준공일자'].astype('str')
        data['준공연도'] = data['준공일자'].str[:4]
        data['준공연도'] = data['준공연도'].astype(np.dtype("int64"))
    
        # 총 면적
        data['총면적'] = (data['전용면적'] + data['공용면적']) * data['전용면적별세대수'] 
        return data
    apart = add_var(data)
    apart
    def remove_var(data, col):
        data.drop(col, axis=1, inplace=True)
        return data
    remove_col = ['단지명', '단지내주차면수', '준공일자']
    apart = remove_var(apart, remove_col)
    
    def data01_process(data):
        data01 = data[['단지코드', '총세대수', '지역', '준공연도', '건물형태', '난방방식', '승강기설치여부']]
        len_1 = len(data01)
        data01 = data01.drop_duplicates()
        data01.reset_index()
        len_2 = len(data01)
        print(f'number of before data01 processing: {len_1}')
        print(f'number of after data01 processing: {len_2}')
        return data01
    data01 = data01_process(apart)

    def data02_process1(data, col1, col2):
        data02 = data[['단지코드', '총면적', '전용면적별세대수', '전용면적', '공용면적', '임대보증금', '임대료']]
        df_area = data02.groupby(col1, as_index=False)[col2].sum()
        return df_area
    df_area = data02_process1(apart, '단지코드', '총면적')

    def data02_process2(data, col1, col2):
        data02 = data[['단지코드', '총면적', '전용면적별세대수', '전용면적', '공용면적', '임대보증금', '임대료']]
        bins = [10, 30, 40, 50, 60, 70, 80, 200]
        labels = [f'면적{bins[i]}_{bins[i+1]}' for i in range(0, len(bins)-1)]
        data02['전용면적구간'] = pd.cut(data02[col2], bins = bins, labels = labels, right=False)
        tmp = data02.groupby([col1, '전용면적구간'], as_index = False)['전용면적구간'].value_counts()
        df_pivot = tmp.pivot(index=col1, columns='전용면적구간')
        df_pivot[labels] = df_pivot['count'][labels]
        df_pivot = df_pivot.drop('count', axis=1)
        df_pivot[col1] = list(df_pivot.index)
        df_pivot.reset_index(drop=True, inplace=True)
        temp_ = pd.DataFrame()
        temp_['단지코드'] = df_pivot['단지코드']
        temp_[labels] = df_pivot[labels]
        df_pivot = temp_
        return df_pivot
    data02 = data[['단지코드', '총면적', '전용면적별세대수', '전용면적', '공용면적', '임대보증금', '임대료']]
    df_pivot = data02_process2(apart, '단지코드', '전용면적')
    data02.groupby(['단지코드', '임대보증금'], as_index = False)['임대보증금'].mean()
    def data02_process3(data):
        data02 = data[['단지코드', '총면적', '전용면적별세대수', '전용면적', '공용면적', '임대보증금', '임대료']]
        df_rent = data02.groupby('단지코드', as_index = False)['임대보증금'].mean()
        df_rent['임대료'] = data02.groupby('단지코드', as_index = False)['임대료'].mean()['임대료']
        return df_rent
    data02 = data[['단지코드', '총면적', '전용면적별세대수', '전용면적', '공용면적', '임대보증금', '임대료']]
    df_rent = data02_process3(apart)
    temp1 = pd.merge(df_area, df_rent, on='단지코드', how='left')
    temp2 = pd.merge(temp1, df_pivot, how='left', on='단지코드')
    base_data = pd.merge(data01, temp2, how='left', on='단지코드')

    base_data['난방방식'] = base_data['난방방식'].replace({
        '개별가스난방': '개별',
        '개별유류난방': '개별',
        '지역난방': '지역',
        '지역가스난방': '지역',
        '지역유류난방': '지역',
        '중앙가스난방': '중앙',
        '중앙난방': '중앙',
        '중앙유류난방': '중앙'
    })

    base_data['승강기설치여부'] = base_data['승강기설치여부'].replace({
        '전체동 설치': 1,
        '일부동 설치': 0,
        '미설치': 0
    })

    base_data = base_data.drop(columns=['단지코드', '지역'])

    base_data = pd.get_dummies(base_data, columns=['건물형태', '난방방식'], drop_first=True)
    base_data['난방방식_중앙'] = base_data['난방방식_중앙'].astype(int)
    base_data['난방방식_지역'] = base_data['난방방식_지역'].astype(int)
    base_data['건물형태_복도식'] = base_data['건물형태_복도식'].astype(int)
    base_data['건물형태_혼합식'] = base_data['건물형태_혼합식'].astype(int)
    return base_data

In [9]:
# 라이브러리 불러오기
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import koreanize_matplotlib
import seaborn as sns

import joblib
import warnings

warnings.filterwarnings(action='ignore')
%config InlineBackend.figure_format='retina'
apart = pd.read_excel('train.xlsx')
tot(apart)

number of before data01 processing: 1157
number of after data01 processing: 345


,총세대수,준공연도,승강기설치여부,총면적,임대보증금,임대료,면적10_30,면적30_40,면적40_50,면적50_60,면적60_70,면적70_80,면적80_200,건물형태_복도식,건물형태_혼합식,난방방식_중앙,난방방식_지역
0,78,2013,1,6023.7683,5.696200e+07,642930.000000,0,0,0,2,0,0,0,0,0,0,0
1,35,2013,1,1569.1668,6.306200e+07,470100.000000,2,0,0,0,0,0,0,1,0,0,0
2,88,2013,1,7180.1396,7.219000e+07,586540.000000,0,0,0,4,0,0,0,0,0,0,0
3,477,2014,1,47058.9273,1.015167e+08,950305.000000,0,0,0,1,0,2,3,1,0,0,1
4,15,2013,1,543.0268,5.522750e+07,340148.333333,6,0,0,0,0,0,0,1,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
340,1485,1993,1,64622.2500,7.595571e+06,104975.714286,5,1,0,1,0,0,0,1,0,1,0
341,1386,1993,1,57616.8100,8.092875e+06,111848.750000,4,1,0,3,0,0,0,1,0,1,0
342,956,1994,1,37398.7200,9.931000e+06,134540.000000,1,0,0,0,0,0,0,1,0,0,1
343,120,2020,1,5581.8024,2.515500e+06,50040.000000,1,1,0,0,0,0,0,1,0,0,0


In [77]:
apart = pd.read_excel('C:/Users/User/mini_2/mini_2_2일차/test.xlsx')
tot(apart)

number of before data01 processing: 104
number of after data01 processing: 30


,총세대수,준공연도,승강기설치여부,총면적,임대보증금,임대료,면적10_30,면적30_40,면적40_50,면적50_60,면적60_70,면적70_80,면적80_200,건물형태_복도식,건물형태_혼합식,난방방식_중앙,난방방식_지역
0,20,2012,1,766.2736,5.236067e+07,305753.333333,3,0,0,0,0,0,0,1,0,0,0
1,822,2018,1,31396.0944,3.546600e+07,445466.666667,0,0,0,3,0,0,0,0,0,0,1
2,112,2014,1,12450.4308,9.869750e+07,744450.000000,0,0,0,0,0,1,1,0,0,0,1
3,122,2011,1,13081.4772,0.000000e+00,0.000000,0,0,0,0,0,2,4,0,0,0,1
4,262,2011,1,28141.7516,0.000000e+00,0.000000,0,0,0,0,0,1,3,0,0,0,1
5,35,2001,1,3379.7260,0.000000e+00,0.000000,0,0,0,0,0,0,1,0,0,0,0
6,47,2009,1,5154.6660,0.000000e+00,0.000000,0,0,0,0,0,1,3,0,0,0,0
7,152,2008,1,17049.8200,0.000000e+00,0.000000,0,0,0,0,0,0,3,0,0,0,1
8,73,2005,1,7845.7626,0.000000e+00,0.000000,0,0,0,0,0,0,2,0,0,0,1
9,571,1998,1,33959.3318,1.670100e+07,215150.000000,0,1,2,0,0,0,0,1,0,0,1


In [75]:
# 파일 불러오기
abcabcabc = joblib.load('C:/Users/User/mini_2/mini_2_2일차/test.xlsx')

UnpicklingError: persistent IDs in protocol 0 must be ASCII strings

In [13]:
abcabc.head(3)

,총세대수,지역,준공연도,승강기설치여부,실차량수,전용면적별세대수,전용면적,공용면적,전용면적구간,총면적,...,50-60,60-70,70-80,80-200,임대보증금,임대료,난방방식_중앙난방,난방방식_지역난방,건물형태_복도식,건물형태_혼합식
0,78,1,2013,1,109,35,51.89,19.2603,4,6023.7683,...,78,0,0,0,56962000.0,642930.0,0,0,0,0
1,78,1,2013,1,109,43,59.93,22.2446,4,6023.7683,...,78,0,0,0,56962000.0,642930.0,0,0,0,0
2,35,1,2013,1,35,26,27.75,16.5375,1,1569.1668,...,0,0,0,0,63062000.0,470100.0,0,0,1,0


In [15]:
abcabc.columns

Index(['총세대수', '지역', '준공연도', '승강기설치여부', '실차량수', '전용면적별세대수', '전용면적', '공용면적',
       '전용면적구간', '총면적', '10-30', '30-40', '40-50', '50-60', '60-70', '70-80',
       '80-200', '임대보증금', '임대료', '난방방식_중앙난방', '난방방식_지역난방', '건물형태_복도식',
       '건물형태_혼합식'],
      dtype='object')

In [57]:
abcabc2 = abcabc.drop(columns = ['10-30', '30-40', '40-50', '50-60', '60-70', '70-80','80-200', '총면적','전용면적구간'])

In [25]:
abcabc2.head(3)

,총세대수,지역,준공연도,승강기설치여부,실차량수,전용면적별세대수,전용면적,공용면적,임대보증금,임대료,난방방식_중앙난방,난방방식_지역난방,건물형태_복도식,건물형태_혼합식
0,78,1,2013,1,109,35,51.89,19.2603,56962000.0,642930.0,0,0,0,0
1,78,1,2013,1,109,43,59.93,22.2446,56962000.0,642930.0,0,0,0,0
2,35,1,2013,1,35,26,27.75,16.5375,63062000.0,470100.0,0,0,1,0


In [59]:
# 라이브러리 불러오기
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

from sklearn.metrics import *

In [61]:
target = '실차량수'
x = abcabc2.drop(columns = target)
y = abcabc2.loc[:, target]

In [63]:
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.3, random_state=1)

In [65]:
# 선언하기
from sklearn.ensemble import RandomForestRegressor
model = RandomForestRegressor(max_depth=10)
# 학습하기
model.fit(x_train, y_train)

# 예측하기
y_pred = model.predict(x_test)

# 평가하기
r2_score(y_test,y_pred)


0.9224783801429529

In [71]:
abcabc2.head(20)

,총세대수,지역,준공연도,승강기설치여부,실차량수,전용면적별세대수,전용면적,공용면적,임대보증금,임대료,난방방식_중앙난방,난방방식_지역난방,건물형태_복도식,건물형태_혼합식
0,78,1,2013,1,109,35,51.89,19.2603,5.696200e+07,642930.000000,0,0,0,0
1,78,1,2013,1,109,43,59.93,22.2446,5.696200e+07,642930.000000,0,0,0,0
2,35,1,2013,1,35,26,27.75,16.5375,6.306200e+07,470100.000000,0,0,1,0
3,35,1,2013,1,35,9,29.08,17.3302,6.306200e+07,470100.000000,0,0,1,0
4,88,1,2013,1,88,7,59.47,21.9462,7.219000e+07,586540.000000,0,0,0,0
5,88,1,2013,1,88,6,59.58,21.9868,7.219000e+07,586540.000000,0,0,0,0
6,88,1,2013,1,88,29,59.60,21.9942,7.219000e+07,586540.000000,0,0,0,0
7,88,1,2013,1,88,46,59.62,22.0016,7.219000e+07,586540.000000,0,0,0,0
8,477,1,2014,1,943,150,59.96,21.5734,1.015167e+08,950305.000000,0,1,1,0
9,477,1,2014,1,943,49,74.89,26.9451,1.015167e+08,950305.000000,0,1,1,0


In [45]:
abcabc2 = abcabc2.drop(columns = '총세대수')

In [67]:
mae = mean_absolute_error(y_test, y_pred)

# 결과 출력
print(f"Mean Absolute Error (MAE): {mae}")

Mean Absolute Error (MAE): 58.02425949307007
